# Trouver les thèmes des avis sur des médicaments avec LDA

Ce TP est une étude de cas d'analyse sémantique, sur les avis sur des médicaments que les TP précédents ont préparés et appris. Il en reprend le nettoyage, déjà écrit, et se demande de quoi parlent les patientes qui jugent leur contraception, et ce qui tire leurs notes vers le bas. Vous construisez le vocabulaire des avis, y cherchez des thèmes avec l'allocation de Dirichlet latente (LDA) de Spark ML, les nommez d'après leurs mots et leurs avis les plus typiques, croisez la part de chaque thème dans un avis avec sa note, suivez les thèmes d'année en année, et finissez par le choix du nombre de thèmes. Le dernier exercice, facultatif, met le modèle en production.

Chaque exercice s'appuie sur le précédent : faites-les dans l'ordre. Spark tourne ici en mode local, sur les deux cœurs de la machine de Colab : la lecture des données et chaque entraînement de LDA prennent une à deux minutes.

## Installation et données

Les données sont le [Drug Review Dataset](https://archive.ics.uci.edu/dataset/462/drug+review+dataset+drugs+com) (Gräßer et Kallumadi, UCI Machine Learning Repository, 2018, licence CC BY 4.0) : des avis publiés sur Drugs.com de 2008 à 2017, en anglais. Ses auteurs l'ont coupé en deux fichiers, `drugsComTrain_raw.tsv` (84 Mo) et `drugsComTest_raw.tsv` (28 Mo) : ce TP ne prédit rien, il prend les deux. Chaque ligne est un avis : le médicament (`drugName`), l'affection soignée (`condition`), le texte (`review`), la note de 1 à 10 (`rating`), la date (`date`) et le nombre de lecteurs qui ont trouvé l'avis utile (`usefulCount`).

In [ ]:
!pip install pyspark
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/drug-reviews/drugsComTrain_raw.tsv
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/drug-reviews/drugsComTest_raw.tsv

In [ ]:
import html
import os
import sys
import time

from pyspark.ml import PipelineModel
from pyspark.ml.clustering import LDA
from pyspark.ml.feature import CountVectorizer, RegexTokenizer, StopWordsRemover
from pyspark.ml.functions import vector_to_array
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

La session tourne en local sur deux cœurs, avec 4 Go de mémoire pour le driver, et quatre partitions pour les agrégations au lieu des 200 par défaut. Les barres de progression de Spark sont masquées : elles encombreraient les sorties. Les processus Python que Spark lance pour exécuter la fonction Python `decoder` du nettoyage prennent l'interpréteur de ce notebook, `sys.executable` : hors de Colab, Spark prendrait sinon le `python3` du `PATH`, qui peut être d'une autre version de Python que celui du driver, ce que Spark refuse.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (
  SparkSession.builder.appName("avis-themes")
  .master("local[2]")
  .config("spark.driver.memory", "4g")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

## Le nettoyage, repris des TP précédents

Ces fonctions reprennent la solution du TP de préparation. `lire` charge un fichier d'avis (un avis peut tenir sur plusieurs lignes) et le répartit sur quatre partitions. `nettoyer` retire les doublons (un même avis publié sous le nom commercial et sous le nom de la molécule), remplace les conditions invalides et manquantes par `"inconnue"`, décode le texte et convertit la date dans `jour`. L'étiquette `positif` des TP précédents ne sert pas ici : on ne prédit rien.

In [ ]:
cles = ["review", "date", "rating", "usefulCount"]
invalide = F.col("condition").contains("</span>") | F.col("condition").rlike("^[a-z]")
decoder = F.udf(lambda texte: html.unescape(texte[1:-1]), "string")


def lire(fichier):
  return (
    spark.read.csv(
      fichier,
      sep="\t",
      header=True,
      inferSchema=True,
      multiLine=True,
      escape='"',
    )
    .withColumnRenamed("_c0", "id")
    .repartition(4)
  )


def nettoyer(avis):
  return (
    avis.dropDuplicates(cles)
    .withColumn("condition", F.when(~invalide, F.col("condition")))
    .withColumn("review", decoder("review"))
    .withColumn("jour", F.to_date("date", "MMMM d, yyyy"))
    .fillna({"condition": "inconnue"})
  )

## 1. Le corpus : les avis sur une contraception

Lisez et nettoyez les deux fichiers avec `lire` et `nettoyer`, réunissez-les dans `avis` et retirez-y de nouveau les doublons sur `cles` : le nettoyage ne les retire qu'à l'intérieur d'un fichier, et un avis peut se trouver dans les deux, sous l'autre nom du médicament. Gardez dans `contraception` les avis dont la condition est `"Birth Control"`, l'affection la plus fréquente, en cache.

Combien `avis` et `contraception` comptent-ils d'avis ? Affichez la note moyenne de `contraception`, la part de ses notes de 1 à 3, puis le nombre d'avis par note, et les cinq médicaments les plus commentés avec leur note moyenne. Pourquoi chercher les thèmes des avis d'une seule affection plutôt que de tous ?


**Documentation utile**

- [pyspark.sql.DataFrame.unionByName](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.unionByName.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`unionByName` réunit deux `DataFrame` de mêmes colonnes, appariées par leur nom. La part des notes de 1 à 3 est la moyenne d'une condition convertie en entier (`(F.col("rating") <= 3).cast("int")`). Vous devez trouver 128 840 avis en tout.

```python
avis = (
  nettoyer(lire("drugsComTrain_raw.tsv"))
  .unionByName(nettoyer(lire(...)))
  .dropDuplicates(...)
)
contraception = avis.filter(F.col("condition") == ...).cache()
print(avis.count(), contraception.count())
contraception.agg(
  F.avg("rating"), F.avg((F.col("rating") <= 3).cast("int")).alias("part_1_3")
).show()
contraception.groupBy(...).count().orderBy(...).show()
(
  contraception.groupBy("drugName")
  .agg(F.count("*").alias("nb_avis"), F.avg("rating").alias("note"))
  .orderBy(F.desc(...))
  .show(5, truncate=False)
)
```

### Solution

In [ ]:
avis = (
  nettoyer(lire("drugsComTrain_raw.tsv"))
  .unionByName(nettoyer(lire("drugsComTest_raw.tsv")))
  .dropDuplicates(cles)
)
contraception = avis.filter(F.col("condition") == "Birth Control").cache()
print(avis.count(), contraception.count())
contraception.agg(
  F.avg("rating"), F.avg((F.col("rating") <= 3).cast("int")).alias("part_1_3")
).show()
contraception.groupBy("rating").count().orderBy("rating").show()
(
  contraception.groupBy("drugName")
  .agg(F.count("*").alias("nb_avis"), F.avg("rating").alias("note"))
  .orderBy(F.desc("nb_avis"))
  .show(5, truncate=False)
)
# 128 840 avis en tout, dont 19 530 sur une contraception : une note moyenne de 6,08, et
# 30 % de notes de 1 à 3. Les notes se concentrent aux extrémités (3 913 notes de 10,
# 2 991 de 1). L'implant (Etonogestrel, Nexplanon) et des pilules sont les plus
# commentés ; le stérilet hormonal a la meilleure note (7,12) : Mirena, Kyleena, Skyla
# et Liletta sont rangés sous sa molécule, Levonorgestrel, et 1 352 de ses 1 457 avis
# parlent d'un stérilet ou de sa pose. Sur tous les avis, chaque thème serait surtout
# une affection (l'acné, la contraception, la tension) ; sur une seule, les thèmes
# disent ce que les patientes en pensent.

## 2. Le vocabulaire

LDA cherche des thèmes dans les comptes de mots de chaque avis : un mot présent dans presque tous les avis, ou qui ne dit rien du sujet, se retrouverait dans tous les thèmes. Le code suivant prépare deux outils. `lettres` découpe le texte en mots de trois lettres au moins, sans chiffres : les doses et les durées (« 20 », « 3mg ») formeraient sinon leurs propres thèmes. `generiques` liste les mots fréquents qui ne disent rien du sujet (des verbes, des durées), absents de la liste des mots vides de Spark.

In [ ]:
lettres = RegexTokenizer(
  inputCol="review", outputCol="mots", pattern=r"[^a-z']+", minTokenLength=3
)
generiques = (
  "get got like just would since started never taking took take week weeks day "
  "days month months year years one two feel getting back every really also even "
  "still much ago far going made last ever few little almost went time"
).split()

Les noms des médicaments sont aussi à retirer : chaque marque formerait son thème, sans rien dire de ce qu'en pensent les patientes. Découpez les noms des médicaments de `contraception` avec `lettres`, et gardez leurs mots distincts dans la liste Python `noms_medicaments`. Construisez ensuite dans `sans_mots_vides` un `StopWordsRemover` de `mots` vers `mots_utiles` qui retire les mots vides anglais de Spark, `generiques` et `noms_medicaments`.

Entraînez sur les mots utiles de `contraception`, dans `vectoriseur`, un `CountVectorizer` de `mots_utiles` vers `comptes` : 5 000 mots au plus, présents dans au moins 10 avis et dans au plus un tiers d'entre eux (`maxDF`). Gardez son vocabulaire dans `vocabulaire`, et dans `comptes` les colonnes `id`, `rating`, `jour` et `comptes` des avis, en cache. Combien de noms de médicaments, et de mots dans le vocabulaire ? Affichez ses quinze premiers mots : pourquoi `pill` et `period` n'y sont-ils pas ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`lettres` lit la colonne `review` : renommez `drugName` en `review` avant de le découper. `F.explode` fait une ligne de chaque mot d'une liste, `distinct` garde chacun une fois, et `collect` rapatrie les lignes sur le driver. `StopWordsRemover.loadDefaultStopWords("english")` rend la liste des mots vides anglais, une liste Python à laquelle ajouter les deux autres. `maxDF` est une part des avis quand il est inférieur à 1 (`maxDF=0.33`), un nombre d'avis sinon ; `minDF` de même.

```python
noms_medicaments = [
  ligne["mot"]
  for ligne in lettres.transform(contraception.select(F.col(...).alias("review")))
  .select(F.explode("mots").alias("mot"))
  .distinct()
  .collect()
]
sans_mots_vides = StopWordsRemover(
  inputCol="mots",
  outputCol="mots_utiles",
  stopWords=StopWordsRemover.loadDefaultStopWords("english") + ... + ...,
)
contraception_mots = sans_mots_vides.transform(lettres.transform(contraception))
vectoriseur = CountVectorizer(
  inputCol="mots_utiles", outputCol="comptes", vocabSize=..., minDF=..., maxDF=...
).fit(contraception_mots)
vocabulaire = vectoriseur.vocabulary
comptes = (
  vectoriseur.transform(contraception_mots)
  .select("id", "rating", "jour", "comptes")
  .cache()
)
print(len(noms_medicaments), len(vocabulaire), comptes.count())
print(vocabulaire[:15])
```

### Solution

In [ ]:
noms_medicaments = [
  ligne["mot"]
  for ligne in lettres.transform(
    contraception.select(F.col("drugName").alias("review"))
  )
  .select(F.explode("mots").alias("mot"))
  .distinct()
  .collect()
]
sans_mots_vides = StopWordsRemover(
  inputCol="mots",
  outputCol="mots_utiles",
  stopWords=StopWordsRemover.loadDefaultStopWords("english")
  + generiques
  + noms_medicaments,
)
contraception_mots = sans_mots_vides.transform(lettres.transform(contraception))
vectoriseur = CountVectorizer(
  inputCol="mots_utiles", outputCol="comptes", vocabSize=5000, minDF=10, maxDF=0.33
).fit(contraception_mots)
vocabulaire = vectoriseur.vocabulary
comptes = (
  vectoriseur.transform(contraception_mots)
  .select("id", "rating", "jour", "comptes")
  .cache()
)
print(len(noms_medicaments), len(vocabulaire), comptes.count())
print(vocabulaire[:15])
# 136 mots de noms de médicaments, et 3 380 mots dans le vocabulaire. Les plus fréquents
# annoncent déjà les sujets : le poids (weight, gain), les règles (periods, bleeding,
# cramps, spotting), l'acné, l'humeur et la libido (mood, sex). pill, period, birth et
# control sont chacun dans plus d'un tiers des avis : maxDF les retire, car ils seraient
# en tête de tous les thèmes sans en distinguer aucun.

## 3. Les thèmes

Entraînez sur `comptes`, dans `modele_defaut`, une LDA à 8 thèmes (`k`) avec l'optimiseur `online`, 30 itérations, 10 % des avis tirés à chaque itération (`subsamplingRate`) et la graine 42, en mesurant la durée de l'entraînement. `modele_defaut.transform(comptes)` ajoute à chaque avis la colonne `topicDistribution` : la part de chaque thème dans l'avis, un vecteur de 8 valeurs de somme 1. Affichez la part moyenne de chaque thème dans les avis : que remarquez-vous ?

Entraînez ensuite dans `modele_lda` la même LDA, avec un `learningOffset` de 10, et comparez ses parts moyennes à celles de `modele_defaut`. Que fait `learningOffset`, et pourquoi le baisser ? `describeTopics` rend, pour chaque thème, les indices de ses mots les plus probables dans le vocabulaire (`termIndices`) et leurs probabilités (`termWeights`) : affichez les huit premiers mots de chaque thème de `modele_lda`.

Les thèmes, et leurs numéros, dépendent de la graine, de la version de Spark et du partitionnement des données : les vôtres peuvent différer de ceux des solutions. Reconnaissez-les à leurs mots.

In [ ]:
# Votre code ici

### Conseils supplémentaires

La colonne des vecteurs se donne par `featuresCol="comptes"`. `vector_to_array` convertit le vecteur `topicDistribution` en tableau, dont `F.col("parts")[j]` donne l'élément `j` : une liste en compréhension passe la moyenne de chaque élément au même `agg`. Une fonction calcule ces moyennes pour les deux modèles, et `setLearningOffset` change le réglage d'une `LDA` déjà construite. Le taux d'apprentissage de l'optimiseur `online` à l'itération $t$ vaut $(\tau_0 + t)^{-\kappa}$, $\tau_0$ étant `learningOffset` (1 024 par défaut) et $\kappa$ `learningDecay` (0,51). `describeTopics(8)` rend un `DataFrame` de quelques lignes : `collect` le rapatrie, et `vocabulaire[i]` donne le mot d'indice `i`.

```python
def parts_moyennes(modele):
  parts = modele.transform(comptes).select(vector_to_array(...).alias("parts"))
  moyennes = parts.agg(*[F.avg(...) for j in range(modele.getK())])
  return [round(p, 3) for p in moyennes.first()]


lda = LDA(
  k=...,
  optimizer=...,
  maxIter=...,
  subsamplingRate=...,
  seed=42,
  featuresCol="comptes",
)
debut = time.time()
modele_defaut = lda.fit(comptes)
print(time.time() - debut, parts_moyennes(modele_defaut))
debut = time.time()
modele_lda = lda.setLearningOffset(...).fit(comptes)
print(time.time() - debut, parts_moyennes(modele_lda))
for ligne in modele_lda.describeTopics(8).collect():
  print(ligne["topic"], " ".join(vocabulaire[i] for i in ...))
```

### Solution

In [ ]:
def parts_moyennes(modele):
  parts = modele.transform(comptes).select(
    vector_to_array("topicDistribution").alias("parts")
  )
  moyennes = parts.agg(*[F.avg(F.col("parts")[j]) for j in range(modele.getK())])
  return [round(p, 3) for p in moyennes.first()]


lda = LDA(
  k=8,
  optimizer="online",
  maxIter=30,
  subsamplingRate=0.1,
  seed=42,
  featuresCol="comptes",
)
debut = time.time()
modele_defaut = lda.fit(comptes)
print(time.time() - debut, parts_moyennes(modele_defaut))
debut = time.time()
modele_lda = lda.setLearningOffset(10).fit(comptes)
print(time.time() - debut, parts_moyennes(modele_lda))
for ligne in modele_lda.describeTopics(8).collect():
  print(ligne["topic"], " ".join(vocabulaire[i] for i in ligne["termIndices"]))
# Environ une minute par entraînement sur deux cœurs. Avec le learningOffset par
# défaut, un thème prend en moyenne 0,80 de chaque avis (il en domine 85 %), et
# plusieurs autres presque rien : LDA n'a pas fini d'apprendre. learningOffset est le
# τ0 du taux d'apprentissage : à 1 024, celui de la première itération vaut
# 1 025^-0,51 ≈ 0,03, les premiers lots comptent à peine, et 30 itérations ne
# suffisent pas. À 10, il part de 11^-0,51 ≈ 0,29, et les huit thèmes se partagent
# les avis (de 0,085 à 0,165 en moyenne). Leurs mots annoncent déjà des sujets
# (l'humeur, le poids, la libido, la pose d'un stérilet, les saignements), que
# l'exercice suivant nomme.

### Nommer les thèmes

Appliquez `modele_lda` à `comptes` dans `themes`, avec le vecteur `topicDistribution` converti en tableau dans la colonne `parts`, en cache. Pour chaque thème, affichez ses huit premiers mots, puis la note et le début (300 caractères) de ses trois avis les plus typiques, ceux où sa part est la plus grande. Donnez un nom à chaque thème, d'après ses mots et ses avis. Que vous apprennent les avis que les mots ne disaient pas ?

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Le texte des avis est dans `contraception` : joignez-le à `themes` par `id`, l'identifiant de chaque avis. `orderBy(F.desc(F.col("parts")[j]))` range les avis par part décroissante du thème `j`, `limit(3)` garde les trois premiers, et `avis["review"][:300]` coupe le texte d'une ligne rapatriée.

```python
themes = modele_lda.transform(comptes).withColumn("parts", vector_to_array(...)).cache()
k = modele_lda.getK()
avis_themes = themes.join(contraception.select("id", "review"), ...)
for ligne in modele_lda.describeTopics(8).collect():
  j = ligne["topic"]
  print(j, " ".join(vocabulaire[i] for i in ligne["termIndices"]))
  for avis in avis_themes.orderBy(...).limit(3).collect():
    print("  ", avis["rating"], avis["review"][:300])
```

#### Solution

In [ ]:
themes = (
  modele_lda.transform(comptes)
  .withColumn("parts", vector_to_array("topicDistribution"))
  .cache()
)
k = modele_lda.getK()
avis_themes = themes.join(contraception.select("id", "review"), "id")
for ligne in modele_lda.describeTopics(8).collect():
  j = ligne["topic"]
  print(j, " ".join(vocabulaire[i] for i in ligne["termIndices"]))
  for avis in avis_themes.orderBy(F.desc(F.col("parts")[j])).limit(3).collect():
    print("  ", avis["rating"], avis["review"][:300])
# 0, l'humeur et la peau (emotional, moody, acne, skin : des pilules qui font
# pleurer ou éclaircissent la peau) ; 1, la prise de poids (weight, gain, pounds),
# racontée ou niée (« no weight gain ») ; 2, la libido et le moral (sex, drive,
# depression, anxiety) ; 3, peu d'effets secondaires (side, effects, reviews, love :
# « very little side effects », « no bad experiences ») ; 4, des règles changées
# sous implant ou stérilet (periods, light, spotting, love : des patientes plutôt
# satisfaites) ; 5, la pose d'un stérilet (pain, insertion, iud, painful) ; 6, les
# saignements (bleeding, heavy, spotting, stopped) ; 7, les complications (pain,
# blood, doctor, removed, severe : grossesse sous stérilet, kyste, urgences). Les
# mots seuls trompaient : side et effects sont en tête du thème 3 parce que ses avis
# disent n'en avoir presque pas, et rien ne disait que le thème 4 était celui de
# l'implant et des stérilets. Les noms sont les vôtres : LDA ne donne que des mots,
# et des parts.

## 4. Les thèmes et les notes

Pour chaque thème de `themes`, affichez sa part moyenne dans les avis et la note moyenne des avis pondérée par cette part. Rangez ensuite chaque avis sous son thème dominant, le plus grand de ses `parts`, dans la colonne `theme`, et affichez par thème le nombre d'avis, leur note moyenne et la part des notes de 1 à 3, par note croissante. Quels thèmes vont avec les mauvaises notes ? Une pose douloureuse fait-elle une mauvaise note ?


**Documentation utile**

- [pyspark.sql.functions.array_position](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.array_position.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`F.col("parts")[j]` donne la part du thème `j`. La note pondérée du thème `j` est $\sum p_j \times \text{note} / \sum p_j$. L'indice du plus grand élément d'un tableau s'obtient en SQL par `array_position(parts, array_max(parts)) - 1` (`array_position` compte à partir de 1).

```python
mesures = []
for j in range(k):
  part = F.col("parts")[j]
  mesures += [
    F.avg(part).alias(f"part_{j}"),
    (F.sum(part * F.col("rating")) / F.sum(...)).alias(f"note_{j}"),
  ]
ligne = themes.agg(*mesures).first()
for j in range(k):
  print(j, round(ligne[f"part_{j}"], 3), round(ligne[f"note_{j}"], 2))
(
  themes.withColumn("theme", F.expr(...))
  .groupBy("theme")
  .agg(
    F.count("*").alias("nb_avis"),
    F.avg("rating").alias("note"),
    F.avg((F.col("rating") <= 3).cast("int")).alias("part_1_3"),
  )
  .orderBy(...)
  .show()
)
```

### Solution

In [ ]:
mesures = []
for j in range(k):
  part = F.col("parts")[j]
  mesures += [
    F.avg(part).alias(f"part_{j}"),
    (F.sum(part * F.col("rating")) / F.sum(part)).alias(f"note_{j}"),
  ]
ligne = themes.agg(*mesures).first()
for j in range(k):
  print(j, round(ligne[f"part_{j}"], 3), round(ligne[f"note_{j}"], 2))
(
  themes.withColumn("theme", F.expr("array_position(parts, array_max(parts)) - 1"))
  .groupBy("theme")
  .agg(
    F.count("*").alias("nb_avis"),
    F.avg("rating").alias("note"),
    F.avg((F.col("rating") <= 3).cast("int")).alias("part_1_3"),
  )
  .orderBy("note")
  .show()
)
# Les complications (7, note pondérée de 4,56), l'humeur et la peau (0, 5,05), la libido
# et le moral (2, 5,09) et les saignements (6, 5,23) tirent les notes vers le bas, sous
# la moyenne de 6,08 : 58 % des avis dominés par les complications ont une note de 1 à
# 3, contre 6 % de ceux qui parlent de règles changées sous implant ou stérilet (4,
# 7,80). La pose d'un stérilet (5) est douloureuse, mais bien notée (7,44) : une
# douleur d'un moment ne fait pas la note. Un thème va avec les mauvaises notes, il ne
# les cause pas forcément : une patiente qui a un problème écrit plus volontiers.

## 5. Les thèmes dans le temps

Affichez, pour chaque année de 2009 à 2017 (2008 ne compte que quelques dizaines d'avis), le nombre d'avis, la part moyenne de chaque thème et la note moyenne, par année. Quels thèmes gagnent du terrain, lesquels en perdent ? La note moyenne suit-elle ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`F.year("jour")` donne l'année d'une date. Une liste en compréhension construit une moyenne par thème, toutes passées au même `agg`.

```python
(
  themes.withColumn("annee", F.year(...))
  .filter(...)
  .groupBy("annee")
  .agg(
    F.count("*").alias("nb_avis"),
    *[F.round(F.avg(F.col("parts")[j]), 3).alias(f"t{j}") for j in range(k)],
    F.round(F.avg("rating"), 2).alias("note"),
  )
  .orderBy("annee")
  .show()
)
```

### Solution

In [ ]:
(
  themes.withColumn("annee", F.year("jour"))
  .filter("annee >= 2009")
  .groupBy("annee")
  .agg(
    F.count("*").alias("nb_avis"),
    *[F.round(F.avg(F.col("parts")[j]), 3).alias(f"t{j}") for j in range(k)],
    F.round(F.avg("rating"), 2).alias("note"),
  )
  .orderBy("annee")
  .show()
)
# De 2014 à 2017, la part moyenne de la pose d'un stérilet passe de 5 % à 13 %, celle
# des complications de 6 % à 10 %, quand la prise de poids recule de 22 % à 13 %. La
# note moyenne tombe de 7,18 à 5,44, mais le nombre d'avis triple en 2015 (de 1 335 à
# 3 830) : le corpus lui-même change (qui écrit, sur quel contraceptif). Les thèmes
# décrivent ce qui se dit, pas pourquoi cela évolue.

## 6. Choisir le nombre de thèmes

Coupez `comptes` en deux parties, 80 % dans `entrainement` et 20 % dans `validation` (graine 42), en cache. Pour $k$ de 4, 8 et 12, entraînez sur `entrainement` une LDA aux réglages de `modele_lda` (exercice 3), et affichez la durée de l'entraînement, la perplexité (`logPerplexity`) sur les deux parties et les cinq premiers mots de chaque thème.

Quel $k$ la perplexité de validation préfère-t-elle ? Les thèmes de ce $k$ sont-ils plus lisibles que ceux de $k = 8$ ? Que faudrait-il pour choisir $k$ ?


**Documentation utile**

- [pyspark.ml.clustering.LDAModel.logPerplexity](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.clustering.LDAModel.html#pyspark.ml.clustering.LDAModel.logPerplexity)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`randomSplit([0.8, 0.2], seed=42)` rend les deux parties. La perplexité est l'exponentielle de l'opposé de la log-vraisemblance moyenne par mot ; `logPerplexity` en rend le logarithme, une borne calculée par l'approximation variationnelle de LDA : plus elle est basse, mieux le modèle prédit les mots des avis. Chaque entraînement prend environ une minute.

```python
entrainement, validation = comptes.randomSplit(..., seed=42)
entrainement.cache()
validation.cache()
for nb_themes in [4, 8, 12]:
  debut = time.time()
  modele = LDA(
    k=nb_themes,
    optimizer="online",
    maxIter=30,
    subsamplingRate=0.1,
    learningOffset=10,
    seed=42,
    featuresCol="comptes",
  ).fit(...)
  print(
    nb_themes,
    round(time.time() - debut),
    round(modele.logPerplexity(...), 3),
    round(modele.logPerplexity(...), 3),
  )
  for ligne in modele.describeTopics(5).collect():
    print("  ", " ".join(vocabulaire[i] for i in ligne["termIndices"]))
```

### Solution

In [ ]:
entrainement, validation = comptes.randomSplit([0.8, 0.2], seed=42)
entrainement.cache()
validation.cache()
for nb_themes in [4, 8, 12]:
  debut = time.time()
  modele = LDA(
    k=nb_themes,
    optimizer="online",
    maxIter=30,
    subsamplingRate=0.1,
    learningOffset=10,
    seed=42,
    featuresCol="comptes",
  ).fit(entrainement)
  print(
    nb_themes,
    round(time.time() - debut),
    round(modele.logPerplexity(entrainement), 3),
    round(modele.logPerplexity(validation), 3),
  )
  for ligne in modele.describeTopics(5).collect():
    print("  ", " ".join(vocabulaire[i] for i in ligne["termIndices"]))
# À ce budget d'entraînement, la perplexité de validation préfère k = 4 (7,070,
# contre 7,219 pour k = 8 et 7,318 pour k = 12), et elle est pire sur la validation
# que sur l'entraînement. Celle de l'entraînement monte aussi avec k (6,885, 6,937,
# 6,966), quand on l'attendrait en baisse, plus de thèmes collant mieux aux avis :
# logPerplexity est une borne variationnelle, pas la vraisemblance exacte, et à budget
# égal (30 itérations sur 10 % des avis) un grand k converge sans doute moins (cause
# probable, non mesurée). Les thèmes de k = 4 mêlent la libido et les saignements, et
# k = 12 en invente un sans sens (begin, cheaper, function, wise). La perplexité
# mesure la prédiction des mots, pas la lisibilité des thèmes : k se choisit à la
# lecture des mots et des avis les plus typiques de chaque thème, relus par quelqu'un
# qui connaît le domaine, la perplexité écartant seulement les k qui prédisent
# nettement moins bien.

## 7. Les thèmes en production (facultatif)

En production, le modèle s'entraîne rarement, et chaque nuit range les nouveaux avis sous ses thèmes, sans réentraînement. Assemblez dans `modele_themes` un `PipelineModel` des quatre étapes déjà prêtes, `lettres`, `sans_mots_vides`, `vectoriseur` et `modele_lda`, sauvegardez-le dans le dossier `modele-themes`, puis rechargez-le dans `modele_recharge`. Appliquez-le à quatre avis inventés, en anglais, dans un `DataFrame` d'une colonne `review`, et affichez pour chacun son thème dominant et sa part : une pose douloureuse, une prise de poids, des saignements qui font retirer le contraceptif, et un dernier qui dit n'avoir pas pris de poids (« No weight gain at all, and I love it »). Chaque avis tombe-t-il sous le thème attendu ? Pourquoi aucune étape n'est-elle réentraînée ?


**Documentation utile**

- [pyspark.ml.PipelineModel](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.PipelineModel.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Un `PipelineModel` se construit directement à partir d'étapes déjà prêtes, sans `fit` : `PipelineModel(stages=[...])`. `write().overwrite().save(chemin)` remplace une sauvegarde précédente, si vous relancez la cellule, et `PipelineModel.load(chemin)` la relit. `spark.createDataFrame([("...",), ...], ["review"])` construit un `DataFrame` d'une colonne à partir de tuples d'un élément. Le thème dominant se calcule comme à l'exercice 4.

```python
modele_themes = PipelineModel(stages=[...])
modele_themes.write().overwrite().save(...)
modele_recharge = PipelineModel.load(...)
nouveaux_avis = spark.createDataFrame(
  [
    ("The insertion was very painful, but I love my IUD now.",),
    (...,),
    (...,),
    ("No weight gain at all, and I love it.",),
  ],
  ["review"],
)
(
  modele_recharge.transform(nouveaux_avis)
  .withColumn("parts", vector_to_array("topicDistribution"))
  .withColumn("theme", F.expr(...))
  .select("review", "theme", F.round(F.array_max("parts"), 2).alias("part"))
  .show(truncate=False)
)
```

### Solution

In [ ]:
modele_themes = PipelineModel(
  stages=[lettres, sans_mots_vides, vectoriseur, modele_lda]
)
modele_themes.write().overwrite().save("modele-themes")
modele_recharge = PipelineModel.load("modele-themes")
nouveaux_avis = spark.createDataFrame(
  [
    ("The insertion was very painful, but I love my IUD now.",),
    ("I gained 15 pounds in three months on this pill.",),
    ("Constant bleeding and spotting, so I had it removed.",),
    ("No weight gain at all, and I love it.",),
  ],
  ["review"],
)
(
  modele_recharge.transform(nouveaux_avis)
  .withColumn("parts", vector_to_array("topicDistribution"))
  .withColumn("theme", F.expr("array_position(parts, array_max(parts)) - 1"))
  .select("review", "theme", F.round(F.array_max("parts"), 2).alias("part"))
  .show(truncate=False)
)
# Les trois premiers avis tombent sous la pose d'un stérilet (5, part de 0,62), la
# prise de poids (1, 0,83) et les saignements (6, 0,86). Le quatrième, qui dit n'avoir
# pas pris de poids, tombe lui aussi sous la prise de poids (1, 0,83) : un sac de mots
# ignore la négation. Rien ne se réentraîne : lettres et sans_mots_vides n'apprennent
# rien, vectoriseur et modele_lda le sont déjà, et transform seul calcule les parts.
# Le vocabulaire et les thèmes restent ainsi les mêmes d'une nuit à l'autre, et les
# parts se comparent ; un mot absent du vocabulaire est ignoré.

## Fin

Arrêtez la session, qui libère les exécuteurs.

In [ ]:
spark.stop()